### Treinamento do Modelo

Pipeline de Treinamento utilizada:
- Pré-Tratamento de dados.
- MICE(Preencher valores faltantes).
- Trigonométrico: Para features ciclicas.
- Capping + Outlier Indicator: Para tratar os outliers e manter a informação de que esses são outliers no Dataset.
- Modelo: Random Forest, Boosting, Bagging.

Esse notebok, serve para criar as Feature Table. Para entendermos quais Features Tables serão criadas, devemos entender quais técnicas de Feature Enginnering serão usadas.
- CyclicalFeatures.
- InputCapping.
- CappingWithIndicator.
- DateTime.


**FeatureTables:**
A quantidade de FeatureTables criadas será de acordo com o seguinte problema fatorial. Para as seguintes coluans do meu dataset:

- Data. 
- Colunas com Outliers.

Existirão as seguintes possibilidades:

- Data -> DateTime .
- Data -> CyclicalFeatures.
- Colunas com Outliers -> Input Capping.
- Colunas com Outliers -> CappingWithIndicator.


Logo existem duas possibilidades, iremos então criar 4 tabelas combinando cada possibilidade.

In [0]:
from itertools import combinations

featlist = {'without_standard','without_outliertreatment','standard','featurecyclical','inputcapping','cappingwithindicator','datetime'}
combinations = combinations(featlist, 3)

def returnFunction(name):
    """
    Return a function according to the name passed
    args:
        name: name of function will be returned.
    """
    if name == 'without_standard' or name == 'without_outliertreatment':
        return 'passthrough'
    elif name == 'standard':
        return StandardScaler
    elif name == 'featurecyclical':
        return FeatureCyclic
    elif name == 'inputcapping':
        return CappingTransformer
    elif name == 'cappingwithindicator':
        return OutlierIndicatorTransformer
    elif name == 'datetime':
        return FeatureDateExtractor


for combination in combinations:
    #Cria a Pipeline---------------------------------------------------------------
    pipeline = Pipeline([
        ('imputer', MiceImputerTransformer()),
        (combination[0], returnFunction(combination[0])()),
        (combination[1], returnFunction(combination[1])()),
        (combination[2], returnFunction(combination[2])())])
    print(pipeline.steps)
    
    #Cria a FeatureTable-----------------------------------------------------------





In [0]:
%pip install -r ../requirements.txt


In [0]:
%restart_python

In [0]:
import pandas as pd
df = pd.read_csv('/Volumes/catalog_samarco/storage_datalake/dbw-otimizacao/pluma_multiclasse_5krows/df_reduzida.csv')

df

In [0]:
import sys

import os
from src.feature_utils import qtdColunasNulas,PreTraining


%load_ext autoreload
%autoreload 2
sys.path.append(os.path.abspath('..'))
y,X = PreTraining(df)

In [0]:
X

In [0]:
colunas_outliers = qtdColunasNulas(X).columns

In [0]:
Y = pd.DataFrame({"y":y})
Y['y'].unique()

In [0]:
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from src.feature_utils import (
    MiceImputerTransformer, 
    FeatureCyclic, 
    CappingTransformer, 
    OutlierIndicatorTransformer
)

# Definição do Pipeline Completo
pipeline_industrial = Pipeline([
    # 1. Primeiro imputamos valores faltantes (MICE)
    ('imputer', MiceImputerTransformer(max_iter=10)),
    
    # 2. Marcamos outliers (baseado nos dados já preenchidos)
    ('outlier_flags', OutlierIndicatorTransformer(cols_to_check=colunas_outliers)),
    
    # 3. Tratamos outliers
    ('capping', CappingTransformer(cols_to_cap=colunas_outliers)),
    
    # 4. Features de Tempo Cíclico
    ('cyclic', FeatureCyclic(col_hora='Hora.1')),
    
    # 5. Modelo Final
    ('model', XGBRegressor())
])

# Quando você roda fit, o MICE aprende só com X_train
# Quando o pipeline roda no X_test, ele usa o conhecimento do treino para preencher o teste
pipeline_industrial.fit(X_train, y_train)

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder # <--- NOVO: Import necessário
import pandas as pd
import numpy as np

# Importa suas classes personalizadas do feature_utils
from src.feature_utils import (
    MiceImputerTransformer, 
    FeatureCyclic, 
    CappingTransformer, 
    OutlierIndicatorTransformer,
    qtdColunasNulas
)

# --- 0. Pré-processamento do TARGET (y) ---
# O XGBoost EXIGE que as classes sejam 0, 1, 2, 3... sequenciais.
# Se suas classes são [0, 4, 30...], precisamos codificá-las.
print("Codificando variáveis alvo (y)...")
le = LabelEncoder()
y_encoded = le.fit_transform(y) # Transforma [0, 4, 30] em [0, 1, 2]

print(f"Classes originais: {le.classes_}")
print(f"Classes transformadas: {np.unique(y_encoded)}")

# --- 1. Divisão Treino vs Teste ---
print("\nDividindo dados em Treino e Teste...")
# Note que agora usamos 'y_encoded' no lugar de 'y'
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded 
)

# --- 2. Definição das Colunas ---
try:
    colunas_outliers = qtdColunasNulas(X).columns.tolist()
except:
    colunas_outliers = None 

# --- 3. Preparação do Pipeline Base ---
preprocessor = Pipeline([
    # Mantive max_iter=100 para evitar o Warning de não convergência do MICE
    ('imputer', MiceImputerTransformer(max_iter=100, random_state=42)), 
    ('outlier_flags', OutlierIndicatorTransformer(cols_to_check=colunas_outliers)),
    ('capping', CappingTransformer(cols_to_cap=colunas_outliers)),
    ('cyclic', FeatureCyclic(col_hora='Hora.1'))
])

# --- 4. Dicionário de Modelos ---
modelos = {
    "XGBoost": XGBClassifier(
        n_jobs=-1,
        objective='multi:softprob',
        random_state=42,
        # Agora o número de classes bate com a sequência 0, 1, 2...
        num_class=len(np.unique(y_train)) 
    ),
    
    "RandomForest": RandomForestClassifier(
        n_jobs=-1, 
        random_state=42,
        class_weight='balanced'
    ),
    
    "Bagging": BaggingClassifier(
        n_jobs=-1, 
        random_state=42
    )
}

# --- 5. Loop de Avaliação ---
resultados = {}
print(f"\nIniciando avaliação com {len(y_train)} amostras de treino...\n")

for nome, modelo in modelos.items():
    print(f"🔄 Treinando {nome}...")
    
    pipeline_completo = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelo)
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    try:
        scores = cross_val_score(
            pipeline_completo, 
            X_train, 
            y_train, 
            cv=cv, 
            scoring='f1_weighted', 
            n_jobs=-1
        )
        
        media_f1 = scores.mean()
        resultados[nome] = media_f1
        print(f"✅ {nome}: F1-Score Médio (CV) = {media_f1:.4f}")
        
    except Exception as e:
        print(f"❌ Falha ao treinar {nome}: {e}")

# --- 6. Escolha e Treino Final ---
if resultados:
    melhor_modelo_nome = max(resultados, key=resultados.get)
    print(f"\n🏆 Campeão: {melhor_modelo_nome} (F1: {resultados[melhor_modelo_nome]:.4f})")

    # Treina o campeão
    pipeline_vencedor = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelos[melhor_modelo_nome])
    ])

    pipeline_vencedor.fit(X_train, y_train)

    # Avaliação final
    from sklearn.metrics import accuracy_score
    y_pred = pipeline_vencedor.predict(X_test)
    score_teste = accuracy_score(y_test, y_pred)
    
    print(f"🚀 Acurácia Final no Test Set (Hold-out): {score_teste:.4f}")
    
    # DICA FINAL: Como recuperar o valor original da previsão?
    # Exemplo: O modelo previu classe '1'. O que isso significa?
    # Resposta: le.inverse_transform([1]) -> Vai retornar '4' (ou o valor original correspondente)
    print("Pipeline pronto para registro no MLflow!")
else:
    print("Nenhum modelo funcionou.")

In [0]:
import re
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Importa suas classes personalizadas do feature_utils
# Certifique-se de que o arquivo src/feature_utils.py está salvo com as alterações anteriores
from src.feature_utils import (
    MiceImputerTransformer, 
    FeatureCyclic, 
    CappingTransformer, 
    OutlierIndicatorTransformer,
    qtdColunasNulas
)

# ==============================================================================
# 1. PREPARAÇÃO E LIMPEZA INICIAL (CRÍTICO PARA XGBOOST)
# ==============================================================================

# A. Codificando variáveis alvo (y) para 0, 1, 2...
print("1. Codificando variáveis alvo (y)...")
le = LabelEncoder()
y_encoded = le.fit_transform(y) # Transforma [0, 4, 30] em [0, 1, 2]

print(f"   Classes originais: {le.classes_}")
print(f"   Classes transformadas: {np.unique(y_encoded)}")

# B. Sanitização dos nomes das colunas (X)
# XGBoost quebra se houver '[', ']' ou '<' nos nomes das colunas
print("2. Sanitizando nomes das colunas para o XGBoost...")
regex = re.compile(r"\[|\]|<", re.IGNORECASE)

X.columns = [
    regex.sub("_", col) if isinstance(col, str) else str(col) 
    for col in X.columns
]
# print(f"   Colunas limpas: {X.columns.tolist()}") # Descomente para verificar

# ==============================================================================
# 2. DIVISÃO TREINO VS TESTE
# ==============================================================================
print("\n3. Dividindo dados em Treino e Teste...")
# stratify=y_encoded garante que as classes raras apareçam em ambos os sets
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded 
)

# ==============================================================================
# 3. DEFINIÇÃO DO PIPELINE
# ==============================================================================

# Tenta identificar colunas que precisam de imputação automaticamente
try:
    colunas_outliers = qtdColunasNulas(X).columns.tolist()
except:
    colunas_outliers = None 

# Pipeline de Pré-processamento
preprocessor = Pipeline([
    # max_iter=100 para garantir que o MICE converja e não dê erro
    ('imputer', MiceImputerTransformer(max_iter=100, random_state=42)), 
    ('outlier_flags', OutlierIndicatorTransformer(cols_to_check=colunas_outliers)),
    ('capping', CappingTransformer(cols_to_cap=colunas_outliers)),
    ('cyclic', FeatureCyclic(col_hora='Hora.1'))
])

# Dicionário de Modelos
modelos = {
    "XGBoost": XGBClassifier(
        n_jobs=-1,
        objective='multi:softprob',
        random_state=42,
        num_class=len(np.unique(y_encoded)) # Define total de classes
    ),
    
    "RandomForest": RandomForestClassifier(
        n_jobs=-1, 
        random_state=42,
        class_weight='balanced'
    ),
    
    "Bagging": BaggingClassifier(
        n_jobs=-1, 
        random_state=42
    )
}

# ==============================================================================
# 4. LOOP DE AVALIAÇÃO (CROSS-VALIDATION)
# ==============================================================================
resultados = {}
print(f"\n4. Iniciando avaliação com {len(y_train)} amostras de treino...\n")

for nome, modelo in modelos.items():
    print(f"🔄 Treinando {nome}...")
    
    pipeline_completo = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelo)
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    try:
        # Avaliação com F1-Weighted (Melhor para multiclasse desbalanceado)
        scores = cross_val_score(
            pipeline_completo, 
            X_train, 
            y_train, 
            cv=cv, 
            scoring='f1_weighted', 
            n_jobs=-1
        )
        
        media_f1 = scores.mean()
        resultados[nome] = media_f1
        print(f"   ✅ {nome}: F1-Score Médio (CV) = {media_f1:.4f}")
        
    except Exception as e:
        print(f"   ❌ Falha ao treinar {nome}: {e}")

# ==============================================================================
# 5. ESCOLHA E TREINO FINAL
# ==============================================================================
if resultados:
    melhor_modelo_nome = max(resultados, key=resultados.get)
    print(f"\n🏆 Campeão: {melhor_modelo_nome} (F1: {resultados[melhor_modelo_nome]:.4f})")

    # Recria o pipeline do vencedor
    pipeline_vencedor = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelos[melhor_modelo_nome])
    ])

    # Treina com TODO o X_train (Fit Final)
    print(f"🚀 Treinando pipeline final do {melhor_modelo_nome}...")
    pipeline_vencedor.fit(X_train, y_train)

    # Avaliação no Test Set (Dados nunca vistos)
    y_pred = pipeline_vencedor.predict(X_test)
    score_teste = accuracy_score(y_test, y_pred)
    
    print(f"📊 Acurácia Final no Test Set (Hold-out): {score_teste:.4f}")
    
    # Exemplo de como recuperar o valor real da classe
    exemplo_pred = y_pred[0]
    valor_real = le.inverse_transform([exemplo_pred])[0]
    print(f"ℹ️ Exemplo: Classe Predita {exemplo_pred} corresponde ao valor original '{valor_real}'")
    
    print("\n✅ Pipeline pronto para registro no MLflow!")
else:
    print("\n❌ Nenhum modelo foi treinado com sucesso. Verifique os erros.")

In [0]:
import re
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier # <--- NOVO: Import da MLP
from sklearn.preprocessing import StandardScaler # <--- NOVO: Obrigatório para MLP
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Importa suas classes personalizadas
from src.feature_utils import (
    MiceImputerTransformer, 
    FeatureCyclic, 
    CappingTransformer, 
    OutlierIndicatorTransformer,
    qtdColunasNulas
)

# ==============================================================================
# 1. PREPARAÇÃO E LIMPEZA
# ==============================================================================
print("1. Codificando variáveis alvo (y)...")
le = LabelEncoder()
y_encoded = le.fit_transform(y) 

print(f"   Classes transformadas: {np.unique(y_encoded)}")

print("2. Sanitizando nomes das colunas...")
regex = re.compile(r"\[|\]|<", re.IGNORECASE)
X.columns = [regex.sub("_", col) if isinstance(col, str) else str(col) for col in X.columns]

# ==============================================================================
# 2. DIVISÃO TREINO VS TESTE
# ==============================================================================
print("\n3. Dividindo dados...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded 
)

# ==============================================================================
# 3. DEFINIÇÃO DO PIPELINE
# ==============================================================================
try:
    colunas_outliers = qtdColunasNulas(X).columns.tolist()
except:
    colunas_outliers = None 

# Pipeline Base (Feature Engineering)
# Adicionei o StandardScaler no final, pois a MLP precisa de dados na mesma escala (média 0, desvio 1)
preprocessor = Pipeline([
    ('imputer', MiceImputerTransformer(max_iter=100, random_state=42)), 
    ('outlier_flags', OutlierIndicatorTransformer(cols_to_check=colunas_outliers)),
    ('capping', CappingTransformer(cols_to_cap=colunas_outliers)),
    ('cyclic', FeatureCyclic(col_hora='Hora.1')),
    ('scaler', StandardScaler()) # <--- CRUCIAL PARA A MLP FUNCIONAR
])

# Dicionário de Modelos
modelos = {
    "XGBoost": XGBClassifier(
        n_jobs=-1,
        objective='multi:softprob',
        random_state=42,
        num_class=len(np.unique(y_encoded))
    ),
    
    "RandomForest": RandomForestClassifier(
        n_jobs=-1, 
        random_state=42,
        class_weight='balanced'
    ),
    
    "Bagging": BaggingClassifier(
        n_jobs=-1, 
        random_state=42
    ),
    
    # --- NOVA REDE NEURAL (MLP) ---
    "MLP (Neural Net)": MLPClassifier(
        hidden_layer_sizes=(128, 64), # Arquitetura Funil (Captura -> Condensa)
        activation='relu',            # Padrão industrial
        solver='adam',                # Melhor otimizador para datasets médios/grandes
        alpha=0.0001,                 # Regularização L2 leve
        batch_size='auto',
        learning_rate='adaptive',     # Diminui o passo se estagnar
        max_iter=500,                 # Dá tempo de convergir
        early_stopping=True,          # Para antes de overfitar
        validation_fraction=0.1,      # Usa 10% do treino interno para checar overfitting
        random_state=42
        # Nota: MLPClassifier não aceita n_jobs=-1 diretamente, ele usa paralelismo interno via BLAS
    )
}

# ==============================================================================
# 4. AVALIAÇÃO
# ==============================================================================
resultados = {}
print(f"\n4. Iniciando avaliação com {len(y_train)} amostras...\n")

for nome, modelo in modelos.items():
    print(f"🔄 Treinando {nome}...")
    
    pipeline_completo = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelo)
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    try:
        # n_jobs=-1 no cross_val_score paraleliza os FOLDS (Treina 5 MLPs ao mesmo tempo)
        scores = cross_val_score(
            pipeline_completo, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1
        )
        
        media_f1 = scores.mean()
        resultados[nome] = media_f1
        print(f"   ✅ {nome}: F1-Score Médio (CV) = {media_f1:.4f}")
        
    except Exception as e:
        print(f"   ❌ Falha ao treinar {nome}: {e}")

# ==============================================================================
# 5. RESULTADO FINAL
# ==============================================================================
if resultados:
    melhor_modelo_nome = max(resultados, key=resultados.get)
    print(f"\n🏆 Campeão: {melhor_modelo_nome} (F1: {resultados[melhor_modelo_nome]:.4f})")

    pipeline_vencedor = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelos[melhor_modelo_nome])
    ])

    print(f"🚀 Treinando pipeline final do {melhor_modelo_nome}...")
    pipeline_vencedor.fit(X_train, y_train)

    y_pred = pipeline_vencedor.predict(X_test)
    score_teste = accuracy_score(y_test, y_pred)
    
    print(f"📊 Acurácia Final no Test Set: {score_teste:.4f}")
    print("\n✅ Pipeline pronto para registro no MLflow!")
else:
    print("\n❌ Nenhum modelo treinado.")

### Tunando o melhor modelo

Agora que temos o melhor modelo, iremos tunar ele:

In [0]:
from sklearn.model_selection import GridSearchCV
import mlflow
import mlflow.sklearn
import warnings

# Silencia warnings chatos do sklearn/xgboost para limpar o output
warnings.filterwarnings('ignore')

print(f"🚀 Iniciando Otimização de Hiperparâmetros (GridSearch) para o {melhor_modelo_nome}...")

# 1. Recria o Pipeline Base do Vencedor
# Precisamos instanciar um pipeline limpo com o XGBClassifier base
pipeline_otimizacao = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_jobs=-1,
        objective='multi:softprob',
        eval_metric='mlogloss', # Métrica interna de otimização
        num_class=len(np.unique(y_encoded)),
        random_state=42
    ))
])

# 2. Define a Grade de Parâmetros (Anti-Overfitting)
# O prefixo 'classifier__' é OBRIGATÓRIO para acessar o modelo dentro do Pipeline
param_grid = {
    # Controle de Complexidade (Evita decorar os dados)
    'classifier__max_depth': [3, 4, 6],       # Árvores rasas generalizam melhor
    'classifier__min_child_weight': [1, 5],   # Peso mínimo para criar nova folha (maior = mais conservador)
    
    # Robustez (Amostragem)
    'classifier__subsample': [0.8],           # Treina com 80% das linhas por árvore
    'classifier__colsample_bytree': [0.8],    # Treina com 80% das colunas por árvore
    
    # Velocidade de Aprendizado vs Quantidade
    'classifier__learning_rate': [0.05, 0.1], # Passo de aprendizado
    'classifier__n_estimators': [100, 200],   # Número de árvores
    
    # Regularização (L2)
    'classifier__reg_lambda': [1.0, 1.5]      # Penalidade extra para evitar pesos explosivos
}

# 3. Configura o GridSearch
grid_search = GridSearchCV(
    estimator=pipeline_otimizacao,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), # Mesma validação rigorosa
    scoring='f1_weighted', # Foco no equilíbrio das classes
    n_jobs=-1,             # Usa todo o poder do cluster
    verbose=1              # Mostra progresso simplificado
)

# 4. Executa a Busca (Isso pode levar alguns minutos)
print("⏳ Buscando a melhor combinação (pode demorar um pouco)...")
grid_search.fit(X_train, y_train)

# 5. Resultados
print("\n" + "="*50)
print(f"🏆 Melhores Parâmetros Encontrados:")
print("="*50)
print(grid_search.best_params_)
print(f"\n📈 Melhor F1-Score (Validação Cruzada): {grid_search.best_score_:.4f}")

# 6. Avaliação Final no Test Set (Hold-out)
melhor_modelo = grid_search.best_estimator_
y_pred_final = melhor_modelo.predict(X_test)
acc_final = accuracy_score(y_test, y_pred_final)

print(f"🚀 Acurácia Final no Test Set (Dados Nunca Vistos): {acc_final:.4f}")

# ==============================================================================
# 7. REGISTRO NO MLFLOW (Padrão Industrial)
# ==============================================================================
# O MLflow no Databricks captura automaticamente, mas vamos garantir o registro explícito
# do melhor modelo com a assinatura correta.

print("\n📦 Registrando no MLflow...")
from mlflow.models import infer_signature

# Define o experimento (opcional, se não definir vai para o padrão do notebook)
# mlflow.set_experiment("/Users/seu_email/pluma_experiment")

with mlflow.start_run(run_name="XGBoost_GridSearch_Best") as run:
    # A. Logar Hiperparâmetros Vencedores
    mlflow.log_params(grid_search.best_params_)
    
    # B. Logar Métricas
    mlflow.log_metric("f1_weighted_cv", grid_search.best_score_)
    mlflow.log_metric("accuracy_test", acc_final)
    
    # C. Inferir Assinatura (Schema de Entrada/Saída)
    # Isso garante que o Model Serving saiba validar os dados depois
    signature = infer_signature(X_train, melhor_modelo.predict(X_train))
    
    # D. Logar o Pipeline Completo (Feature Eng + Modelo)
    mlflow.sklearn.log_model(
        sk_model=melhor_modelo,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(5), # Exemplo para a documentação da API
        registered_model_name="main.default.pluma_pricing_classifier" # Nome no Unity Catalog
    )
    
    print(f"✅ Modelo salvo e registrado com sucesso! Run ID: {run.info.run_id}")
    print("👉 Agora você pode ir em 'Serving' e criar o Endpoint usando este modelo.")